# Fit a model

**The job.** Same data, two questions. Predict a number, and predict a label.

The point here is that both are the *same graph*. Only the last two steps
change. Swapping regression for classification is a change of route, not a
rewrite.

Everything is numpy. No scikit-learn. The maths is short enough to read, and
you can see there is nothing hidden in it.

**In:** a generated dataset.
**Out:** scores for both jobs, and predictions.
**Files:** metrics.json, predictions.csv.

In [ ]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

## The input

800 rows. Three numeric columns and one category. The number we predict depends
on all of them plus noise. The label is just "is the number above average" —
so a model that predicts the number well should classify well too.

In [ ]:
import numpy as np

rng = np.random.default_rng(11)
N = 800

size = rng.normal(70, 18, N).round(1)
age = rng.integers(1, 40, N)
rooms = rng.integers(1, 6, N)
area = rng.choice(["north", "south", "central"], N, p=[0.4, 0.35, 0.25])

premium = np.select([area == "central", area == "south"], [95.0, 40.0], 0.0)
price = (28 * size - 1.7 * age + 12 * rooms + premium
         + rng.normal(0, 45, N)).round(1)
above = (price > np.median(price)).astype(int)

print(f"{N} rows")
print(f"{'size':>8}{'age':>6}{'rooms':>7}{'area':>9}{'price':>10}{'above':>7}")
for i in range(5):
    print(f"{size[i]:>8.1f}{age[i]:>6}{rooms[i]:>7}{area[i]:>9}{price[i]:>10.1f}{above[i]:>7}")
print(f"\nprice: min {price.min():.0f}  median {np.median(price):.0f}  max {price.max():.0f}")

## The steps

Load, split, then two independent feature passes — numbers and the category —
that meet at the assemble step. Then fit and score.

Numbers and categories are genuinely independent. Neither waits on the other.
Drawn as a list that fact disappears.

In [ ]:
nodes = [
    node("load.arrays",  "data.read",    [],                  [("out", "Frame")]),
    node("split.random", "data.split",   [("in", "Frame")],   [("train", "Frame"), ("valid", "Frame")]),
    node("num.standard", "feature.numeric",     [("in", "Frame")], [("out", "Matrix")]),
    node("num.raw",      "feature.numeric",     [("in", "Frame")], [("out", "Matrix")]),
    node("cat.onehot",   "feature.categorical", [("in", "Frame")], [("out", "Matrix")]),
    node("assemble.hstack","feature.assemble",  [("numeric", "Matrix"), ("categorical", "Matrix")], [("out", "Matrix")]),
    node("fit.leastsquares","model.fit", [("in", "Matrix")],  [("out", "Model")]),
    node("fit.logistic",    "model.fit", [("in", "Matrix")],  [("out", "Model")]),
    node("fit.ridge",       "model.fit", [("in", "Matrix")],  [("out", "Model")]),
    node("score.regression","model.score",[("in", "Model")],  [("out", "Score")]),
    node("score.classifier","model.score",[("in", "Model")],  [("out", "Score")]),
]

stages = [
    stage("load",     "Load the data",     [],                [("out", "Frame")], "data.read", ["load.arrays"]),
    stage("split",    "Hold some back",    [("in", "Frame")], [("train", "Frame"), ("valid", "Frame")], "data.split", ["split.random"]),
    stage("numeric",  "Scale the numbers", [("in", "Frame")], [("out", "Matrix")], "feature.numeric",     ["num.standard", "num.raw"]),
    stage("category", "Encode the area",   [("in", "Frame")], [("out", "Matrix")], "feature.categorical", ["cat.onehot"]),
    stage("assemble", "Put them together", [("numeric", "Matrix"), ("categorical", "Matrix")], [("out", "Matrix")], "feature.assemble", ["assemble.hstack"]),
    stage("fit",      "Fit a model",       [("in", "Matrix")], [("out", "Model")], "model.fit",   ["fit.leastsquares", "fit.logistic", "fit.ridge"]),
    stage("score",    "Score it",          [("in", "Model")],  [("out", "Score")], "model.score", ["score.regression", "score.classifier"]),
]

edges = [Edge("load", "split"), Edge("split", "numeric", from_port="train"),
         Edge("split", "category", from_port="train"),
         Edge("numeric", "assemble", to_port="numeric"),
         Edge("category", "assemble", to_port="categorical"),
         Edge("assemble", "fit"), Edge("fit", "score")]

bench = build("Fit a model", "Predict a number, and predict a label.",
              stages, nodes, edges)
print("layers:", bench.layers())
print("routes:", bench.route_count(), "— two ways to fit, two ways to score")

In [ ]:
viz.dag(bench)

## The code

`np.linalg.lstsq` for the regression. Plain gradient descent for the logistic
one. Both are a few lines, and both are doing the real thing.

In [ ]:
def load_arrays():
    return {"size": size, "age": age, "rooms": rooms, "area": area,
            "price": price, "above": above}

def split_random(**kw):
    """Its own seeded generator, not the shared one.

    This started out drawing from the module-level `rng`, which advances every
    time anything uses it. Two effects, both bad. Re-running the same plan gave
    a different score, so the plan's own claim to be deterministic was false.
    And worse, comparing four routes below would have scored each one on a
    *different* split — which is not a comparison of models at all.
    """
    frame = kw["in"]
    order = np.random.default_rng(2024).permutation(N)
    cut = int(N * 0.75)
    take = lambda idx: {k: v[idx] for k, v in frame.items()}
    return {"train": take(order[:cut]), "valid": take(order[cut:])}

def num_standard(**kw):
    """Centre and scale, using the training numbers only."""
    frame = kw["in"]
    cols = np.column_stack([frame["size"], frame["age"], frame["rooms"]]).astype(float)
    mean, sd = cols.mean(0), cols.std(0)
    return {"matrix": (cols - mean) / sd, "mean": mean, "sd": sd,
            "target": frame["price"], "label": frame["above"]}

def cat_onehot(**kw):
    """One column per area. Three areas, three columns."""
    frame = kw["in"]
    areas = ["north", "south", "central"]
    matrix = np.column_stack([(frame["area"] == a).astype(float) for a in areas])
    return {"matrix": matrix, "areas": areas}

def assemble_hstack(**kw):
    numeric, categorical = kw["numeric"], kw["categorical"]
    X = np.column_stack([np.ones(len(numeric["matrix"])),
                         numeric["matrix"], categorical["matrix"]])
    return {"X": X, "y": numeric["target"], "label": numeric["label"],
            "names": ["bias", "size", "age", "rooms", *categorical["areas"]]}

def fit_leastsquares(**kw):
    data = kw["in"]
    weights, *_ = np.linalg.lstsq(data["X"], data["y"], rcond=None)
    return {"kind": "regression", "weights": weights, "data": data}

def num_raw(**kw):
    """No scaling at all. Kept as a real option so the search has something to
    reject on evidence rather than on somebody's opinion."""
    frame = kw["in"]
    cols = np.column_stack([frame["size"], frame["age"], frame["rooms"]]).astype(float)
    return {"matrix": cols, "mean": np.zeros(3), "sd": np.ones(3),
            "target": frame["price"], "label": frame["above"]}

def fit_ridge(**kw):
    """Least squares with a small penalty on big weights."""
    data = kw["in"]
    X, y = data["X"], data["y"]
    penalty = 1.0 * np.eye(X.shape[1])
    penalty[0, 0] = 0.0                       # never penalise the bias
    weights = np.linalg.solve(X.T @ X + penalty, X.T @ y)
    return {"kind": "regression", "weights": weights, "data": data}

def fit_logistic(**kw):
    """Gradient descent. 400 steps is plenty for this."""
    data = kw["in"]
    X, y = data["X"], data["label"].astype(float)
    w = np.zeros(X.shape[1])
    for _ in range(400):
        p = 1 / (1 + np.exp(-X @ w))
        w -= 0.5 * (X.T @ (p - y)) / len(y)
    return {"kind": "classification", "weights": w, "data": data}

def score_regression(**kw):
    m = kw["in"]
    X, y = m["data"]["X"], m["data"]["y"]
    pred = X @ m["weights"]
    ss_res = float(((y - pred) ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    return {"kind": "regression", "r2": 1 - ss_res / ss_tot,
            "mae": float(np.abs(y - pred).mean()),
            "prediction": pred, "truth": y,
            "weights": dict(zip(m["data"]["names"], m["weights"].round(2)))}

def score_classifier(**kw):
    m = kw["in"]
    X, y = m["data"]["X"], m["data"]["label"]
    prob = 1 / (1 + np.exp(-X @ m["weights"]))
    pred = (prob > 0.5).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    return {"kind": "classification",
            "accuracy": float((pred == y).mean()),
            "precision": tp / (tp + fp) if tp + fp else 0.0,
            "recall": tp / (tp + fn) if tp + fn else 0.0,
            "confusion": {"tp": tp, "fp": fp, "tn": tn, "fn": fn},
            "prediction": pred, "truth": y}

runtime = execute.Runtime({
    "load.arrays": load_arrays, "split.random": split_random,
    "num.standard": num_standard, "num.raw": num_raw, "cat.onehot": cat_onehot,
    "fit.ridge": fit_ridge,
    "assemble.hstack": assemble_hstack,
    "fit.leastsquares": fit_leastsquares, "fit.logistic": fit_logistic,
    "score.regression": score_regression, "score.classifier": score_classifier,
})
print("all steps have code:", runtime.missing(
    compile_route(bench, {**{s.id: s.candidates[0] for s in bench.leaf_stages}})) == [])

## Route one: predict the number

In [ ]:
regression = {s.id: s.candidates[0] for s in bench.leaf_stages}
regression["fit"] = "fit.leastsquares"
regression["score"] = "score.regression"

plan_r = compile_route(bench, regression)
run_r = execute.run(plan_r, runtime)
print(run_r.text())

got = run_r.output("score")
print(f"\nR²  {got['r2']:.4f}      mean error {got['mae']:,.1f}")
print("\nwhat it learned:")
for name, weight in got["weights"].items():
    print(f"  {name:<9}{weight:>10.2f}")

The weights line up with how the data was made: `size` is the big driver,
`central` is worth more than `south`, and `age` pulls down. That is a check on
the pipeline, not just on the model.

## Route two: predict the label

Same graph. Two different candidates.

In [ ]:
classification = dict(regression)
classification["fit"] = "fit.logistic"
classification["score"] = "score.classifier"

plan_c = compile_route(bench, classification)
run_c = execute.run(plan_c, runtime)

got_c = run_c.output("score")
print(f"accuracy {got_c['accuracy']:.3f}   precision {got_c['precision']:.3f}   recall {got_c['recall']:.3f}")
c = got_c["confusion"]
print(f"\n            predicted 0   predicted 1")
print(f"  actual 0 {c['tn']:>12} {c['fp']:>13}")
print(f"  actual 1 {c['fn']:>12} {c['tp']:>13}")
print(f"\nsame graph? {plan_r.layers == plan_c.layers}")
print(f"different plan? {plan_r.digest != plan_c.digest}")

Same layers, different digest. The shape of the work did not change. What ran
inside it did, and the digest proves the two results came from different graphs
so they can never be mixed up later.

## Let the evidence pick, instead of picking yourself

So far I chose the route by hand. Fine for two options. There are now six ways
to predict the number — two ways to handle the numbers, three ways to fit — and
picking by hand stops being a plan.

So run them all and let the measured result decide. Not a prior. Not an opinion
about which model is better. The actual score on this actual data.

In [ ]:
import itertools

numeric_options = bench.stage("numeric").candidates
fit_options = [c for c in bench.stage("fit").candidates if c != "fit.logistic"]

results = []
for numeric, fit in itertools.product(numeric_options, fit_options):
    trial = dict(regression, numeric=numeric, fit=fit, score="score.regression")
    plan_t = compile_route(bench, trial)
    got_t = execute.run(plan_t, runtime)
    if not got_t.ok:
        results.append((numeric, fit, None, None, plan_t.digest, got_t.steps[-1].error))
        continue
    s_t = got_t.output("score")
    results.append((numeric, fit, s_t["r2"], s_t["mae"], plan_t.digest, ""))

print(f"{'numbers':<14}{'model':<18}{'R2':>9}{'mean error':>13}   plan")
for numeric, fit, r2, mae, digest, err in results:
    if r2 is None:
        print(f"{numeric:<14}{fit:<18}{'failed':>9}{'':>13}   {err[:34]}")
    else:
        print(f"{numeric:<14}{fit:<18}{r2:>9.4f}{mae:>13,.1f}   {digest[5:17]}")

## The winner, and how much of the space that took

In [ ]:
ranked = sorted([r for r in results if r[2] is not None], key=lambda r: -r[2])
best = ranked[0]
print(f"best:  {best[0]:<13}+ {best[1]:<18}R2 {best[2]:.4f}   plan {best[4][5:17]}")
print(f"worst: {ranked[-1][0]:<13}+ {ranked[-1][1]:<18}R2 {ranked[-1][2]:.4f}")
print(f"\ngap between best and worst: {best[2] - ranked[-1][2]:.4f} R2")
print("\nEvery number above was measured. None of it was a prior.")

In [ ]:
viz.funnel([
    ("every route in the graph", bench.route_count()),
    ("regression routes",        len(results)),
    ("ran without failing",      len(ranked)),
    ("chosen",                   1),
], title="how the model was picked")

### Read that result honestly

The gap between best and worst is **0.0000**. On this data, with this split,
the choice makes no measurable difference at all.

So the right conclusion is not "num.standard won". It is "this decision does not
matter here, so stop spending time on it". Scaling does nothing for least
squares because least squares is scale-invariant, and the ridge penalty is too
small to bite. Both of those are true facts about the maths, and the measurement
agrees with them.

A search that always announces a winner will always find one. The useful search
tells you when the winner is noise.

Four routes tried, four measured, one picked. Small enough to enumerate, and the
notebook says so rather than implying a bigger search happened.

When the space is too big to enumerate, `browsergraph.search` does this with a
beam and reports how much of the space it covered. It refuses to enumerate a
space it cannot finish, rather than trying and running out of memory — which is
exactly what it used to do.

In [ ]:
best_route = dict(regression, numeric=best[0], fit=best[1])
plan_best = compile_route(bench, best_route)
run_best = execute.run(plan_best, runtime)
print(f"re-ran the winner: R2 {run_best.output('score')['r2']:.4f}"
      f"   same plan: {plan_best.digest == best[4]}")

## Save the results

In [ ]:
import csv

def write_results(workspace, **kw):
    metrics = {"regression": {k: v for k, v in kw["reg"].items()
                              if k not in ("prediction", "truth", "weights")},
               "classification": {k: v for k, v in kw["clf"].items()
                                  if k not in ("prediction", "truth")},
               "regression_weights": {k: float(v) for k, v in kw["reg"]["weights"].items()},
               "plans": {"regression": kw["reg_plan"], "classification": kw["clf_plan"]}}
    (workspace / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))

    path = workspace / "predictions.csv"
    with path.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["actual_price", "predicted_price", "actual_label", "predicted_label"])
        for row in zip(kw["reg"]["truth"], kw["reg"]["prediction"],
                       kw["clf"]["truth"], kw["clf"]["prediction"]):
            writer.writerow([f"{row[0]:.1f}", f"{row[1]:.1f}", int(row[2]), int(row[3])])
    return {"rows": len(kw["reg"]["truth"])}

wrote = write_results(WORK, reg=got, clf=got_c,
                      reg_plan=plan_r.digest, clf_plan=plan_c.digest)
print(wrote)
for name in ("metrics.json", "predictions.csv"):
    path = WORK / name
    print(f"  {name:<18}{path.stat().st_size:>8,} bytes")
print()
print((WORK / "predictions.csv").read_text().splitlines()[0])
for line in (WORK / "predictions.csv").read_text().splitlines()[1:6]:
    print(line)

## What the five notebooks showed

Same library, five jobs that have nothing in common:

1. a web page turned into rows,
2. mixed records forced into one schema,
3. images checked and resized,
4. a messy table cleaned with a record of every change,
5. a model fitted two different ways.

Each one had real input, ran real code, and left files behind. The graph was
written the same way every time, the checks were the same checks, and the
picture was drawn by the same function.